# Tutorial 3: Cell Embedding Generation and Visualization

Welcome to Tutorial 3! This covers **Phase 4: Cell Embedding Generation**.

## Learning Objectives

By the end of this tutorial, you will understand:
1. How to extract cell embeddings from a pretrained model
2. Different pooling strategies (CLS, mean, max)
3. How to visualize embeddings with UMAP and t-SNE
4. How to evaluate embedding quality
5. How embeddings compare to traditional methods (PCA)

## Prerequisites

Complete Tutorial 2 or have a pretrained model checkpoint ready.

## Part 1: Understanding Cell Embeddings

**Cell embeddings** are low-dimensional representations that capture biological information:

- Each cell → **d_model dimensional vector** (e.g., 32 dimensions)
- Similar cells → **nearby in embedding space**
- Different cell types → **separated clusters**

### Three Pooling Strategies:

1. **CLS token** (default): Use the [CLS] token embedding (like BERT)
2. **Mean pooling**: Average all gene embeddings
3. **Max pooling**: Take maximum across gene embeddings

Let's compare them!

In [ ]:
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print(f"PyTorch version: {torch.__version__}")

## Part 2: Loading Pretrained Model and Data

Let's load our pretrained model from Tutorial 2:

In [ ]:
from scgpt_mini.model import TransformerModel
from scgpt_mini.tokenizer import GeneVocab
from scgpt_mini.data import preprocess_adata

# Load data and preprocess
adata = sc.datasets.pbmc3k_processed()  # Use preprocessed version with labels

print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Cell type labels: {adata.obs['louvain'].unique().tolist()}")

In [ ]:
# Preprocess to match training
adata = preprocess_adata(
    adata,
    filter_gene_by_counts=10,
    filter_cell_by_genes=200,
    normalize_total_target=1e4,
    log1p=True,
    subset_hvg=500,
    binning=False,
    inplace=False,
)

print(f"Preprocessed: {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
# Create vocabulary
vocab = GeneVocab(adata.var_names.tolist())

# Load pretrained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_config = {
    "vocab_size": len(vocab),
    "d_model": 32,
    "nhead": 2,
    "num_layers": 2,
    "d_hid": 64,
    "dropout": 0.1,
    "max_seq_len": 1001,
    "value_mode": "continuous",
}

model = TransformerModel(**model_config, vocab=vocab)

# Load checkpoint (adjust path if needed)
checkpoint_path = "./tutorial_output/final_model.pt"
if Path(checkpoint_path).exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded pretrained model from epoch {checkpoint['epoch']}")
else:
    print("⚠️  No checkpoint found. Using randomly initialized model.")

model = model.to(device)
model.eval()
print("Model ready for inference!")

## Part 3: Extracting Cell Embeddings

Now let's extract embeddings using all three pooling strategies:

In [ ]:
from scgpt_mini.tasks import extract_cell_embeddings

# Extract with CLS token (default)
embeddings_cls = extract_cell_embeddings(
    model=model,
    adata=adata,
    vocab=vocab,
    batch_size=64,
    pool_strategy='cls',
    device=device,
)

print(f"CLS embeddings shape: {embeddings_cls.shape}")
print(f"Embedding dimension: {embeddings_cls.shape[1]}")

In [ ]:
# Extract with mean pooling
embeddings_mean = extract_cell_embeddings(
    model=model,
    adata=adata,
    vocab=vocab,
    batch_size=64,
    pool_strategy='mean',
    device=device,
)

# Extract with max pooling
embeddings_max = extract_cell_embeddings(
    model=model,
    adata=adata,
    vocab=vocab,
    batch_size=64,
    pool_strategy='max',
    device=device,
)

print(f"Mean pooling: {embeddings_mean.shape}")
print(f"Max pooling: {embeddings_max.shape}")

In [ ]:
# Store embeddings in AnnData
adata.obsm['X_scgpt_cls'] = embeddings_cls
adata.obsm['X_scgpt_mean'] = embeddings_mean
adata.obsm['X_scgpt_max'] = embeddings_max

print("✅ Embeddings stored in adata.obsm")

## Part 4: Visualizing Embeddings with UMAP

UMAP reduces high-dimensional embeddings to 2D for visualization:

In [ ]:
import umap

# Compute UMAP for all three strategies
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)

umap_cls = reducer.fit_transform(embeddings_cls)
umap_mean = reducer.fit_transform(embeddings_mean)
umap_max = reducer.fit_transform(embeddings_max)

# Store in AnnData
adata.obsm['X_umap_cls'] = umap_cls
adata.obsm['X_umap_mean'] = umap_mean
adata.obsm['X_umap_max'] = umap_max

print("UMAP coordinates computed!")

In [ ]:
# Visualize all three strategies
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cell_types = adata.obs['louvain'].values
unique_types = np.unique(cell_types)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_types)))

for ax, umap_coords, title in zip(
    axes,
    [umap_cls, umap_mean, umap_max],
    ['CLS Token', 'Mean Pooling', 'Max Pooling']
):
    for i, cell_type in enumerate(unique_types):
        mask = cell_types == cell_type
        ax.scatter(
            umap_coords[mask, 0],
            umap_coords[mask, 1],
            c=[colors[i]],
            label=cell_type,
            s=10,
            alpha=0.6,
        )
    
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(f'scGPT-mini ({title})')
    if ax == axes[0]:
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('embeddings_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 5: Comparing to PCA

How do scGPT embeddings compare to traditional PCA?

In [ ]:
# Compute PCA on raw data
from sklearn.decomposition import PCA

data_matrix = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
pca = PCA(n_components=32, random_state=42)
embeddings_pca = pca.fit_transform(data_matrix)

# Compute UMAP on PCA
umap_pca = reducer.fit_transform(embeddings_pca)

print(f"PCA embeddings: {embeddings_pca.shape}")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Compare PCA vs scGPT
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, umap_coords, title in zip(
    axes,
    [umap_pca, umap_cls],
    ['PCA (Traditional)', 'scGPT-mini (CLS)']
):
    for i, cell_type in enumerate(unique_types):
        mask = cell_types == cell_type
        ax.scatter(
            umap_coords[mask, 0],
            umap_coords[mask, 1],
            c=[colors[i]],
            label=cell_type,
            s=15,
            alpha=0.6,
        )
    
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(title)

axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('pca_vs_scgpt.png', dpi=150, bbox_inches='tight')
plt.show()

print("Compare cluster separation!")

## Part 6: Quantitative Evaluation

Let's measure embedding quality with metrics:

1. **Silhouette Score**: How well-separated are clusters? (higher = better)
2. **Adjusted Rand Index (ARI)**: Clustering agreement with labels
3. **k-NN Accuracy**: Label transfer quality

In [ ]:
from scgpt_mini.tasks.embedding import (
    compute_silhouette_score,
    compute_ari,
    knn_accuracy,
)

# Get cell type labels
labels = adata.obs['louvain'].values

# Evaluate all embedding methods
methods = {
    'PCA': embeddings_pca,
    'scGPT (CLS)': embeddings_cls,
    'scGPT (Mean)': embeddings_mean,
    'scGPT (Max)': embeddings_max,
}

results = []

for method_name, embeddings in methods.items():
    # Silhouette score
    silhouette = compute_silhouette_score(embeddings, labels)
    
    # ARI
    ari = compute_ari(embeddings, labels, n_clusters=len(unique_types))
    
    # k-NN accuracy
    knn_acc = knn_accuracy(embeddings, labels, k=5, test_size=0.3, random_state=42)
    
    results.append({
        'Method': method_name,
        'Silhouette': silhouette,
        'ARI': ari,
        'k-NN Accuracy': knn_acc,
    })

# Create DataFrame
results_df = pd.DataFrame(results)
print("\nEmbedding Quality Metrics:")
print(results_df.to_string(index=False))

In [ ]:
# Visualize metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['Silhouette', 'ARI', 'k-NN Accuracy']
colors_bar = ['skyblue', 'lightcoral', 'lightgreen', 'plum']

for ax, metric in zip(axes, metrics):
    ax.bar(
        results_df['Method'],
        results_df[metric],
        color=colors_bar,
        edgecolor='black',
    )
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Method')
    ax.set_xticklabels(results_df['Method'], rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('embedding_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpreting the Metrics:

- **Silhouette > 0.3**: Good cluster separation
- **ARI > 0.5**: Strong clustering agreement
- **k-NN Accuracy > 0.8**: Excellent label transfer

scGPT embeddings should outperform PCA if the model learned meaningful patterns!

## Part 7: Visualizing with t-SNE

t-SNE is another popular dimensionality reduction method:

In [ ]:
from sklearn.manifold import TSNE

# Compute t-SNE for scGPT embeddings
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_coords = tsne.fit_transform(embeddings_cls)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, coords, title in zip(
    axes,
    [umap_cls, tsne_coords],
    ['UMAP', 't-SNE']
):
    for i, cell_type in enumerate(unique_types):
        mask = cell_types == cell_type
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[colors[i]],
            label=cell_type,
            s=15,
            alpha=0.6,
        )
    
    ax.set_xlabel(f'{title} 1')
    ax.set_ylabel(f'{title} 2')
    ax.set_title(f'scGPT Embeddings ({title})')

axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('umap_vs_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 8: Analyzing Embedding Space

Let's understand what the model learned by examining embedding statistics:

In [ ]:
# Distribution of embedding values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all embedding values
axes[0].hist(embeddings_cls.flatten(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Embedding Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Embedding Values')
axes[0].grid(True, alpha=0.3)

# Per-dimension variance
dim_variance = embeddings_cls.var(axis=0)
axes[1].bar(range(len(dim_variance)), dim_variance, edgecolor='black')
axes[1].set_xlabel('Embedding Dimension')
axes[1].set_ylabel('Variance')
axes[1].set_title('Variance per Embedding Dimension')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean embedding value: {embeddings_cls.mean():.4f}")
print(f"Std embedding value: {embeddings_cls.std():.4f}")
print(f"Total variance: {dim_variance.sum():.4f}")

## Part 9: Using Embeddings with Scanpy

Embeddings can be used with scanpy's visualization tools:

In [ ]:
# Use scanpy to visualize
sc.pl.embedding(
    adata,
    basis='umap_cls',
    color='louvain',
    title='scGPT-mini Embeddings',
    frameon=False,
)

# Can also visualize gene expression on embedding
if 'CD3D' in adata.var_names:
    sc.pl.embedding(
        adata,
        basis='umap_cls',
        color='CD3D',
        title='CD3D Expression (T cell marker)',
        frameon=False,
    )

## Summary

In this tutorial, you learned:

✅ **Embedding Extraction**:
- CLS, mean, and max pooling strategies
- Using pretrained models for inference

✅ **Visualization**:
- UMAP and t-SNE for 2D projection
- Comparing different pooling methods
- Integration with scanpy

✅ **Evaluation**:
- Silhouette score for cluster quality
- ARI for clustering agreement
- k-NN accuracy for label transfer

✅ **Comparison**:
- scGPT vs PCA embeddings
- Quantitative and visual comparison

✅ **Analysis**:
- Understanding embedding statistics
- Interpreting what the model learned

## Next Steps

Continue to **Tutorial 4: Cell Type Annotation** to learn how to fine-tune the model for classification!

## Exercises

Try these to deepen your understanding:

1. **Dimension reduction**: Try different UMAP/t-SNE parameters (perplexity, n_neighbors)
2. **Pooling comparison**: Which pooling strategy works best for your data?
3. **Batch effects**: If you have batch information, check if embeddings mix batches well
4. **Custom metrics**: Implement your own embedding quality metric
5. **Visualization**: Create an interactive plot with plotly or bokeh
6. **Gene importance**: Analyze which genes contribute most to embeddings
7. **Trajectory inference**: Use embeddings for pseudotime analysis with scanpy